# Borcea's Conjecture

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order




def ideal_x_n_minus_1_coeffs(n):
  """Returns the coefficients of the polynomial x^n - 1."""
  coeffs = np.zeros(n + 1)
  coeffs[0] = 1  # Coefficient of x^n
  coeffs[-1] = -1  # Constant term
  return coeffs


def coefficient_distance_to_x_n_minus_c_family(roots):
  """Calculates a penalty based on how close the polynomial's coefficients are to the form x^n - c (family of polynomials).

  Args:
    roots: A numpy array of complex numbers representing the roots of a
      polynomial.

  Returns:
    A float representing the penalty.
  """
  coeffs = polynomial_roots_to_coeffs(roots)
  degree = len(coeffs) - 1

  # Initialize penalty (we'll sum squared magnitudes of intermediate
  # coefficients)
  penalty_distance = 0

  # Iterate through intermediate coefficients (from x^(n-1) down to x^1)
  for i in range(1, degree):  # Indices 1 to n-1 correspond to x^(n-1) to x^1
    coeff_index = i  # Index in the coefficient array (coeffs)
    coeff_magnitude_squared = np.abs(coeffs[coeff_index]) ** 2
    penalty_distance += coeff_magnitude_squared

  # We could also penalize deviation of the leading coefficient from 1
  # (optional):
  leading_coeff_penalty = np.abs(coeffs[0] - 1) ** 2
  penalty_distance += leading_coeff_penalty

  return penalty_distance


def polynomial_roots_to_coeffs(roots):
  """Converts roots of a polynomial to its coefficients."""
  return np.poly(roots)


def polynomial_coeffs_to_derivative_coeffs(coeffs):
  """Calculates coefficients of the derivative of a polynomial."""
  derivative_coeffs = np.polyder(coeffs)
  return derivative_coeffs


def polynomial_critical_points(roots):
  """Calculates critical points of a polynomial given its roots."""
  coeffs = polynomial_roots_to_coeffs(roots)
  derivative_coeffs = polynomial_coeffs_to_derivative_coeffs(coeffs)
  critical_points = np.roots(derivative_coeffs)
  return critical_points


def calculate_extremal_family_penalty(
    roots, critical_points, max_min_distance, tolerance=0.75
):
  """Calculates penalty if construction resembles the extremal family."""
  penalty = 0.0
  special_zero_found = False

  for i in range(len(roots)):
    special_zero = roots[i]

    # Calculate distances from the special zero to all critical points
    distances_to_special_zero = np.abs(critical_points - special_zero)

    # Check if all critical points are approximately at max_min_distance from
    # the special zero
    if np.all(np.abs(distances_to_special_zero - max_min_distance) < tolerance):
      special_zero_found = True
      break  # No need to check other roots if one special zero is found

  if special_zero_found:
    penalty = 1.0  # Increased penalty for closer resemblance to extremal family

  return penalty


def evaluate_construction(roots):
  """Evaluates Sendov's conjecture for a given set of roots (optimized).

  Args:
    roots: A numpy array of complex numbers representing the roots of a
      polynomial.

  Returns:
    The score for the given set of roots.
  """
  n = len(roots)
  if n < 15 or n > 30:
    return 0  # Conjecture is for degree n >= 2

  # Check that the max magnitude is not too large
  if np.max(np.abs(roots)) > 20:
    return 0

  # Scale roots so their average modulus is less than 1
  if np.mean(np.abs(roots)) < 0.01:
    return -1
  roots = roots / np.mean(np.abs(roots))

  # Check that at most half of the roots are near the origin
  close_to_origin = np.sum(np.abs(roots) < 0.3)
  if close_to_origin > n / 5:
    return 0

  critical_points = polynomial_critical_points(roots)

  # Vectorized distance calculation and minimum finding
  distances = np.abs(roots[:, np.newaxis] - critical_points)  # shape (n, n-1)
  min_distances = np.min(distances, axis=1)  # shape (n,)
  max_min_distance = np.max(min_distances)

  penalty = calculate_extremal_family_penalty(
      roots, critical_points, max_min_distance
  )

  coeff_unity_distance = coefficient_distance_to_x_n_minus_c_family(roots)
  if coeff_unity_distance < n:
    penalty += 1.0
  else:
    penalty += 0

  return max_min_distance - penalty


############### #Hidden variants of above code


def polynomial_roots_to_coeffs_h(roots):
  """Converts roots of a polynomial to its coefficients."""
  return np.poly(roots)


def polynomial_coeffs_to_derivative_coeffs_h(coeffs):
  """Calculates coefficients of the derivative of a polynomial."""
  derivative_coeffs = np.polyder(coeffs)
  return derivative_coeffs


def polynomial_critical_points_h(roots):
  """Calculates critical points of a polynomial given its roots."""
  coeffs = polynomial_roots_to_coeffs_h(roots)
  derivative_coeffs = polynomial_coeffs_to_derivative_coeffs_h(coeffs)
  critical_points = np.roots(derivative_coeffs)
  return critical_points


def evaluate_sendov_conjecture_h(roots):
  """Evaluates Sendov's conjecture for a given set of roots (optimized).

  Args:
    roots: A numpy array of complex numbers representing the roots of a
      polynomial.

  Returns:
    The score for the given set of roots.
  """
  n = len(roots)
  if n < 15 or n > 30:
    return 0  # Conjecture is for degree n >= 2

  # Check that the max magnitude is not too large
  if np.max(np.abs(roots)) > 20:
    return 0
  # Scale roots so their average modulus is less than 1
  # Check first if dividing by zero
  if np.mean(np.abs(roots)) < 0.01:
    return 0
  roots = roots / np.mean(np.abs(roots))

  # Check that at most half of the roots are near the origin
  close_to_origin = np.sum(np.abs(roots) < 0.3)
  if close_to_origin > n / 5:
    return 0

  critical_points = polynomial_critical_points_h(roots)

  # Vectorized distance calculation and minimum finding
  distances = np.abs(roots[:, np.newaxis] - critical_points)  # shape (n, n-1)
  min_distances = np.min(distances, axis=1)  # shape (n,)
  max_min_distance = np.max(min_distances)

  penalty = calculate_extremal_family_penalty(
      roots, critical_points, max_min_distance
  )
  coeff_unity_distance = coefficient_distance_to_x_n_minus_c_family(roots)
  if coeff_unity_distance < n:
    penalty += 1.0
  else:
    penalty += 0

  return max_min_distance - penalty


def format_feedback_repr(feedback):
  """Formats feedback dictionary for representation as code."""
  formatted_feedback = {}
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      repr_str = repr(value)  # Get repr string (e.g., "array([[...], [...]])")
      cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)  # Clean up

      # Remove the leading "array(" and trailing ")" from repr string, then wrap
      # with "np.array(...)"
      array_content = cleaned_repr_str[
          6:-1
      ]  # Extract content inside "array(...)"

      if np.iscomplexobj(value):
        formatted_feedback[key] = (  # Use extracted content in np.array
            f'np.array({array_content}, dtype=np.complex128)'
        )
      elif not np.issubdtype(value.dtype, np.inexact):
        formatted_feedback[key] = f'np.array({array_content}, dtype=np.float64)'
      else:
        formatted_feedback[key] = f'np.array({array_content})'

    else:
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(
    hypers: Mapping[str, Any],
) -> tuple[dict[str, float], dict[str, str]]:
  """Returns the numerical bound for the polygons if valid, or 0 if invalid."""
  result = {}
  feedback = {}
  del hypers
  best_construction = search_for_best_poly()
  result['score'] = evaluate_sendov_conjecture_h(best_construction)

  feedback['best_roots'] = best_construction
  feedback['best_score_found'] = result['score']
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Initial program

"""Finds a function that give the best bound for Sendov's conjecture."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import re
from typing import Any, Callable, Mapping
import scipy.linalg as la
import numpy.polynomial.polynomial as poly

minimize = optimize.minimize



def search_for_best_poly() -> np.ndarray:
  """Function to search for the best construction."""
  roots = np.array([
      0.1 + 0.1j,
      -0.2 - 0.3j,
      0.4,
      -0.5,
      0.3,
      0.1,
      -1,
      0.8 + 0.1j,
      0.9 - 0.1j,
      -0.8j,
      0.7,
      0.1 + 0.2j,
      0.2 - 0.1j,
      0.3 + 0.1j,
      0.4 - 0.1j,
      0.5 - 0.1j,
      0.6 - 0.1j,
      0.7 - 0.1j,
      0.8 - 0.16j,
      0.9 - 0.19j,
      0.543 - 0.1432j,
      0.343 - 0.1234j,
      0.43 - 0.6j,
      0.3 - 0.1j,
      0.6 - 0.1j,
      0.234 - 0.62j,
  ])
  best_roots = roots.copy()
  best_score = evaluate_construction(roots)
  start_time = time.time()
  while time.time() - start_time < np.random.randint(10, 500):
    roots[np.random.randint(2)] += np.random.uniform(-0.1, 0.1)
    score = evaluate_construction(roots)
    if score > best_score:
      best_score = score
      best_roots = roots.copy()
      print(score)
  return best_roots

**Prompt used**

Act as an expert software developer and inequality specialist specializing in creating polynomials with certain properties. You will be trying to find good examples to a complicated analysis problem.
Your task is to generate the sequence of roots of a complex polynomial, that maximizes the following evaluation function:

def evaluate_construction(roots):
  """Evaluates construction for a given set of roots (optimized).

Returns the maximum distance from each root to its closest critical point.
  """
  n = len(roots)
  if n < 15 or n > 50:
    return 0  

# Check that the max magnitude is not too large
  if np.max(np.abs(roots)) > 20:
    return 0

# Scale roots so their average modulus is less than 1
  if np.mean(np.abs(roots)) < 0.01:
    return -1
  roots = roots / np.mean(np.abs(roots))

# Check that at most half of the roots are far from the origin
  close_to_origin = np.sum(np.abs(roots) < 0.1)
  if close_to_origin > n / 2:
    return 0

critical_points = polynomial_critical_points(roots)

# Vectorized distance calculation and minimum finding
  distances = np.abs(roots[:, np.newaxis] - critical_points)  # shape (n, n-1)
  min_distances = np.min(distances, axis=1)  # shape (n,)
  max_min_distance = np.max(min_distances)

penalty = calculate_extremal_family_penalty(
      roots, critical_points, max_min_distance
  )

coeff_unity_distance = coefficient_distance_to_x_n_minus_c_family(roots)
  if coeff_unity_distance < n:
    penalty += 1.0
  else:
    penalty += 0

return max_min_distance - penalty

Your task is to write a search function that searches for the best list of roots. Your function will have 500 seconds to run, and after that it has to have returned the best construction it found. If after 500 seconds it has not returned anything, it will be terminated with negative infinity points. You may freely choose the value of n, the number of zeros. Your n must be at least 15 and at most 50 for it to receive a score.

You may code up any search method you want, and you are allowed to call the evaluate_construction() function as many times as you want. You have access to it, you don't need to code up the evaluate_construction() function.



In [ ]:
#@title Code evolve by AlphaEvolve

def search_for_best_poly() -> np.ndarray:
  """Searches for the best construction using gradient-based optimization."""
  start_time = time.time()
  best_score = -np.inf
  best_roots = None

  for n in range(15, 51):

    # Initialize roots
    angles = np.linspace(0, 2 * np.pi, n - 1, endpoint=False)
    roots = np.array(
        [np.exp(1j * angle) for angle in angles]
    )  # Roots on the unit circle
    roots = np.append(roots, [8.0])  # Add an outlier


    def evaluate_wrapper(x):

      r = x[:n] + 1j * x[n:]
      val = evaluate_construction(r)

      return -val if val is not None else np.inf

    # Gradient-based optimization
    x0 = np.concatenate((roots.real, roots.imag))


    res = minimize(evaluate_wrapper, x0, method='Nelder-Mead', options={'maxiter': 100, 'fatol': 1e-2, 'xatol':1e-2})

    optimized_roots = res.x[:n] + 1j * res.x[n:]
    score = evaluate_construction(optimized_roots)

    if score > best_score:
        best_score = score
        best_roots = optimized_roots
        print(f"New best score for n={n}: {best_score}")

    if time.time() - start_time > 500:
        break


  return best_roots


## What AlphaEvolve found

AlphaEvolve was tasked with finding a counterexample to Borcea's conjecture, focusing on the $p=1$ case. AlphaEvolve proposed various $z^n - nz$ and $z^n - nz^{n-1}$ type constructions. Despite attempts to push AlphaEvolve away from these known polynomial forms by giving it a penalty for constructions too similar to them, it ultimately did not find a counterexample to this conjecture.